# 🎯 Proyecto Integrador: Clasificación de Iris

## Objetivo
Aplicar todo lo aprendido en el curso en un proyecto end-to-end:
- Carga y exploración de datos reales
- Visualización y análisis
- Entrenamiento de múltiples modelos
- Evaluación con K-Fold Cross-Validation
- Comparación de modelos

## Dataset
Iris dataset con 3 especies de flores y 4 características.

In [ ]:
# ==========================================
# CONFIGURACIÓN DEL ENTORNO
# ==========================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sys
from pathlib import Path

# Agregar el directorio raíz al path de manera robusta
project_root = Path.cwd().parent.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Verificar que las utilidades se pueden importar
try:
    from utils.plot_utils import plot_confusion_matrix, plot_decision_boundary
    print("✅ Entorno configurado correctamente")
    print(f"📁 Raíz del proyecto: {project_root}")
except ImportError as e:
    print("❌ Error al importar utilidades")
    print("\n💡 Soluciones:")
    print("   1. Ejecuta 'pip install -e .' desde la raíz del proyecto")
    print("   2. O inicia Jupyter desde la raíz: cd ML-FROM-ZERO-PYTHON && jupyter notebook")
    print(f"\n🔍 Error detallado: {e}")
    raise

# Configuración de visualización
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

## 1. Carga y Exploración de Datos

In [ ]:
# Cargar dataset
data_path = project_root / 'data' / 'iris_extended.csv'
df = pd.read_csv(data_path)

print("📊 Dataset cargado correctamente")
print(f"\nForma: {df.shape}")
print(f"\nPrimeras filas:")
print(df.head())
print(f"\nInformación del dataset:")
print(df.info())
print(f"\nEstadísticas descriptivas:")
print(df.describe())

In [ ]:
# Distribución de clases
print("\n📈 Distribución de especies:")
print(df['species'].value_counts())

plt.figure(figsize=(10, 6))
df['species'].value_counts().plot(kind='bar', color=['#FF6B6B', '#4ECDC4', '#95E1D3'])
plt.title('Distribución de Especies de Iris', fontsize=14, fontweight='bold')
plt.xlabel('Especie')
plt.ylabel('Cantidad')
plt.xticks(rotation=45)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 2. Visualización de Datos

In [ ]:
# Pairplot para ver relaciones entre features
plt.figure(figsize=(12, 10))
sns.pairplot(df, hue='species', markers=['o', 's', 'D'])
plt.suptitle('Relaciones entre Features por Especie', y=1.02, fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Matriz de correlación
plt.figure(figsize=(10, 8))
correlation_matrix = df.drop('species', axis=1).corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, 
            square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Matriz de Correlación de Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n📊 Observaciones:")
print("- Petal length y petal width están altamente correlacionadas")
print("- Sepal length tiene correlación moderada con petal measurements")

## 3. Preparación de Datos

In [ ]:
# Convertir especies a números
species_map = {'setosa': 0, 'versicolor': 1, 'virginica': 2}
df['species_encoded'] = df['species'].map(species_map)

# Separar features y target
X = df[['sepal_length', 'sepal_width', 'petal_length', 'petal_width']].values
y = df['species_encoded'].values

print(f"✅ Datos preparados:")
print(f"   X shape: {X.shape}")
print(f"   y shape: {y.shape}")
print(f"   Clases: {np.unique(y)}")

In [ ]:
# Normalizar features (importante para algunos algoritmos)
X_mean = X.mean(axis=0)
X_std = X.std(axis=0)
X_normalized = (X - X_mean) / X_std

print("✅ Features normalizadas (media=0, std=1)")
print(f"\nMedia después de normalización: {X_normalized.mean(axis=0)}")
print(f"Std después de normalización: {X_normalized.std(axis=0)}")

## 4. Implementación de Modelos

Vamos a usar los modelos implementados en el curso.

In [ ]:
# ==========================================
# MODELO 1: Regresión Logística Multiclase
# ==========================================

class LogisticRegressionMulticlass:
    """
    Regresión Logística Multiclase usando One-vs-Rest.
    """
    
    def __init__(self, learning_rate=0.01, n_iterations=1000):
        self.learning_rate = learning_rate
        self.n_iterations = n_iterations
        self.models = {}
        self.classes = None
    
    def sigmoid(self, z):
        return 1 / (1 + np.exp(-np.clip(z, -500, 500)))
    
    def fit(self, X, y):
        self.classes = np.unique(y)
        n_samples, n_features = X.shape
        
        # Entrenar un modelo binario para cada clase
        for cls in self.classes:
            # Crear etiquetas binarias
            y_binary = (y == cls).astype(int)
            
            # Inicializar parámetros
            w = np.zeros(n_features)
            b = 0
            
            # Gradient Descent
            for _ in range(self.n_iterations):
                z = np.dot(X, w) + b
                y_pred = self.sigmoid(z)
                
                # Gradientes
                dw = (1/n_samples) * np.dot(X.T, (y_pred - y_binary))
                db = (1/n_samples) * np.sum(y_pred - y_binary)
                
                # Actualizar parámetros
                w -= self.learning_rate * dw
                b -= self.learning_rate * db
            
            self.models[cls] = {'w': w, 'b': b}
        
        return self
    
    def predict_proba(self, X):
        probas = np.zeros((X.shape[0], len(self.classes)))
        
        for i, cls in enumerate(self.classes):
            w = self.models[cls]['w']
            b = self.models[cls]['b']
            z = np.dot(X, w) + b
            probas[:, i] = self.sigmoid(z)
        
        return probas
    
    def predict(self, X):
        probas = self.predict_proba(X)
        return self.classes[np.argmax(probas, axis=1)]

print("✅ Modelo 1: Regresión Logística Multiclase definido")

In [ ]:
# ==========================================
# MODELO 2: K-Nearest Neighbors (KNN)
# ==========================================

class KNN:
    """
    K-Nearest Neighbors para clasificación.
    """
    
    def __init__(self, k=3):
        self.k = k
        self.X_train = None
        self.y_train = None
    
    def fit(self, X, y):
        self.X_train = X
        self.y_train = y
        return self
    
    def predict(self, X):
        predictions = []
        
        for x in X:
            # Calcular distancias a todos los puntos de entrenamiento
            distances = np.sqrt(np.sum((self.X_train - x)**2, axis=1))
            
            # Obtener índices de los k vecinos más cercanos
            k_indices = np.argsort(distances)[:self.k]
            
            # Obtener etiquetas de los k vecinos
            k_nearest_labels = self.y_train[k_indices]
            
            # Votación por mayoría
            most_common = np.bincount(k_nearest_labels.astype(int)).argmax()
            predictions.append(most_common)
        
        return np.array(predictions)

print("✅ Modelo 2: K-Nearest Neighbors definido")

In [ ]:
# ==========================================
# K-FOLD CROSS-VALIDATION
# ==========================================

class KFoldCV:
    """
    K-Fold Cross-Validation desde cero.
    """
    
    def __init__(self, n_splits=5, shuffle=True, random_state=None):
        self.n_splits = n_splits
        self.shuffle = shuffle
        self.random_state = random_state
    
    def split(self, X, y=None):
        n_samples = len(X)
        indices = np.arange(n_samples)
        
        if self.shuffle:
            if self.random_state is not None:
                np.random.seed(self.random_state)
            np.random.shuffle(indices)
        
        fold_sizes = np.full(self.n_splits, n_samples // self.n_splits, dtype=int)
        fold_sizes[:n_samples % self.n_splits] += 1
        
        current = 0
        for fold_size in fold_sizes:
            start, stop = current, current + fold_size
            test_indices = indices[start:stop]
            train_indices = np.concatenate([indices[:start], indices[stop:]])
            yield train_indices, test_indices
            current = stop

print("✅ K-Fold Cross-Validation definido")

## 5. Evaluación con K-Fold Cross-Validation

In [ ]:
# Evaluar Regresión Logística
print("🔍 Evaluando Regresión Logística Multiclase...\n")

kfold = KFoldCV(n_splits=5, shuffle=True, random_state=42)
logistic_scores = []

for fold, (train_idx, test_idx) in enumerate(kfold.split(X_normalized, y), 1):
    X_train, X_test = X_normalized[train_idx], X_normalized[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    
    # Entrenar modelo
    model = LogisticRegressionMulticlass(learning_rate=0.1, n_iterations=1000)
    model.fit(X_train, y_train)
    
    # Predecir
    y_pred = model.predict(X_test)
    
    # Calcular accuracy
    accuracy = np.mean(y_pred == y_test)
    logistic_scores.append(accuracy)
    
    print(f"Fold {fold}: Accuracy = {accuracy:.4f}")

print(f"\n📊 Regresión Logística - Accuracy promedio: {np.mean(logistic_scores):.4f} ± {np.std(logistic_scores):.4f}")

In [ ]:
# Evaluar KNN con diferentes valores de k
print("\n🔍 Evaluando K-Nearest Neighbors...\n")

k_values = [3, 5, 7]
knn_results = {}

for k in k_values:
    print(f"\n--- KNN con k={k} ---")
    knn_scores = []
    
    for fold, (train_idx, test_idx) in enumerate(kfold.split(X_normalized, y), 1):
        X_train, X_test = X_normalized[train_idx], X_normalized[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        
        # Entrenar modelo
        model = KNN(k=k)
        model.fit(X_train, y_train)
        
        # Predecir
        y_pred = model.predict(X_test)
        
        # Calcular accuracy
        accuracy = np.mean(y_pred == y_test)
        knn_scores.append(accuracy)
        
        print(f"Fold {fold}: Accuracy = {accuracy:.4f}")
    
    knn_results[k] = knn_scores
    print(f"\n📊 KNN (k={k}) - Accuracy promedio: {np.mean(knn_scores):.4f} ± {np.std(knn_scores):.4f}")

## 6. Comparación de Modelos

In [ ]:
# Preparar datos para visualización
results = {
    'Logistic Regression': logistic_scores,
    'KNN (k=3)': knn_results[3],
    'KNN (k=5)': knn_results[5],
    'KNN (k=7)': knn_results[7]
}

# Boxplot de comparación
plt.figure(figsize=(12, 6))
plt.boxplot(results.values(), labels=results.keys())
plt.ylabel('Accuracy', fontsize=12)
plt.title('Comparación de Modelos - K-Fold Cross-Validation', fontsize=14, fontweight='bold')
plt.grid(axis='y', alpha=0.3)
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

# Tabla de resultados
print("\n" + "="*60)
print("📊 RESUMEN DE RESULTADOS")
print("="*60)
print(f"{'Modelo':<25} {'Mean Accuracy':<20} {'Std':<10}")
print("-"*60)
for model_name, scores in results.items():
    print(f"{model_name:<25} {np.mean(scores):<20.4f} {np.std(scores):<10.4f}")
print("="*60)

In [ ]:
# Determinar mejor modelo
mean_scores = {name: np.mean(scores) for name, scores in results.items()}
best_model = max(mean_scores, key=mean_scores.get)
best_score = mean_scores[best_model]

print(f"\n🏆 MEJOR MODELO: {best_model}")
print(f"   Accuracy promedio: {best_score:.4f}")
print(f"\n✨ ¡Felicitaciones! Has completado un proyecto de ML end-to-end.")

## 7. Conclusiones y Aprendizajes

### Lo que logramos:
1. ✅ Cargamos y exploramos un dataset real
2. ✅ Visualizamos relaciones entre features
3. ✅ Preparamos y normalizamos los datos
4. ✅ Implementamos múltiples modelos desde cero
5. ✅ Evaluamos con K-Fold Cross-Validation
6. ✅ Comparamos resultados y seleccionamos el mejor modelo

### Observaciones:
- Los modelos funcionan bien en este dataset (accuracy > 90%)
- KNN es sensible al valor de k
- La normalización es crucial para el rendimiento
- K-Fold CV proporciona una evaluación más robusta que un simple train/test split

### Próximos pasos:
- Probar con datasets más complejos
- Implementar más modelos (SVM, Random Forest)
- Realizar feature engineering
- Optimizar hiperparámetros con Grid Search

## 🎓 ¡Has completado el curso de ML desde CERO!

Ahora tienes las bases sólidas para:
- Entender cómo funcionan los algoritmos de ML internamente
- Implementar tus propias versiones de algoritmos
- Usar librerías de alto nivel (scikit-learn) con conocimiento profundo
- Continuar aprendiendo algoritmos más avanzados

**¡Sigue practicando y construyendo proyectos!** 🚀